In [5]:
import numpy as np
from pulser import Register, Sequence, Pulse
from pulser.devices import DigitalAnalogDevice
from pulser.waveforms import BlackmanWaveform
from pulser_simulation import QutipEmulator


def build(with_pi_pulse):
    reg = Register({"q0": (0, 0)})
    seq = Sequence(reg, DigitalAnalogDevice)
    seq.declare_channel("raman", "raman_local", initial_target="q0")
    if with_pi_pulse:
        pulse = Pulse.ConstantDetuning(
            BlackmanWaveform(duration=200, area=np.pi / 2), detuning=0, phase=0
        )
    else:
        pulse = Pulse.ConstantDetuning(
            BlackmanWaveform(duration=200, area=np.pi / 2), detuning=0, phase=np.pi / 2
        )
    seq.add(pulse, "raman")
    seq.measure(basis="digital")
    return seq


for with_pi in (False, True):
    res = QutipEmulator.from_sequence(build(with_pi)).run()
    psi = res.get_final_state().full().flatten()
    print("=== Avec impulsion π ===" if with_pi else "=== Sans impulsion ===")
    print("statevector :", np.round(psi, 3))
    print("bitstrings  :", dict(res.sample_final_state(N_samples=1000)))

=== Sans impulsion ===
statevector : [-0.707+0.j  0.707+0.j]
bitstrings  : {'0': 501, '1': 499}
=== Avec impulsion π ===
statevector : [0.   +0.707j 0.707+0.j   ]
bitstrings  : {'1': 506, '0': 494}


/tmp/ipykernel_72506/3885450727.py:26: DeprecationWarning: QutipEmulator is deprecated as of pulser 1.9. Please use QutipBackendV2 instead.
  res = QutipEmulator.from_sequence(build(with_pi)).run()
/tmp/ipykernel_72506/3885450727.py:26: DeprecationWarning: QutipEmulator is deprecated as of pulser 1.9. Please use QutipBackendV2 instead.
  res = QutipEmulator.from_sequence(build(with_pi)).run()


In [ ]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Parameter, ParameterVector
from qiskit.primitives import StatevectorEstimator

q1 = QuantumCircuit(QuantumRegister(1))
q1.rx(np.pi / 2, 0)
q1.ry(np.pi / 2, 0)
estimator = StatevectorEstimator()
result = StatevectorEstimator().run([circuit], shots=epoch).result()